In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Atualizando o chat() para aceitar "tool_choice"
# Isso permite forcar o Claude a usar uma tool especifica (ex: web_search)
# em vez de deixar ele decidir sozinho se pesquisa ou nao.

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None, tool_choice=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if tool_choice:
        params["tool_choice"] = tool_choice

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

In [4]:
web_search_schema = {
    "type":"web_search_20250305",
    "name":"web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
    
}

In [5]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle ? 
    """
)
response = chat(
    messages,
    tools=[web_search_schema],
    tool_choice={"type": "tool", "name": "web_search"}
)
response.content

[ServerToolUseBlock(id='srvtoolu_01LthdWD5pedZ1T4zU46Mtis', caller=None, input={'query': 'best exercise leg muscle growth'}, name='web_search', type='server_tool_use'),
 WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='Er8ICioIEhgCIiQxN2Y2MjhlZi1jMmUyLTQzZGYtODI1Ny01ZjU4YTdjMzJlN2USDNTBwEJun8ObHFeESRoM5YKdyaPQECMdxwbEIjAi9zf7e5YZSzhGRHmklCawgwmaJ58r/nAxZ6wznZYOTZU2EtedUCYvcKs4Xx2pqhUqwgdBpNn8b8cmOqcBN0FeYlQRu7hylOSgl1RNwRpNYw0crRvSy6ZeHNYGA1wCwMRnTRB0LzIowepZMAA0tpmqLWBDR6985vMtlnNW6pYUNA3AcA4Ko79xzAf0BMM9t7YRTohyV44mp02douy8FySldLImZPuNnYCTLhdKoMTmb1v3h52lf4WvvB6ViS4yg5SewhcEZ6qIwT980XfGY/qWSlPgRUfAHQydxjUCRbeTkTJpV9n1JTJAJkqFuaMTb7HppM9obFCIaHwxPzPV0A/FH/UNdcIeqLWHrStc8+cSiFd6x3mk6Imscw/2zeLl+Qcp7CcyaifhQ8OXKWkVY5hgxTzo8YI2v4dMhTypSDOrlZED17YZApn4PdWwwyCyY4dWf1lvD2O56DTIUX5YNfw/Jl6sqrAxZVf3PHl/QNXtIGqFCULTFKj08bcPh4Y5N7f8YWIWOHvjzYQc2w0+IkEXrcYnp+RI3Qoau8VHC0a+LtGFIcKy1RcA+RqLKTllDAu5Z5GWd1QyMK/f0ARPkbVGfQURQPN/DY4WLnUr0wp5